# Chat — Jupyter integration
This notebook provides an interactive chat UI that sends messages to a Chat API (OpenAI-compatible). It demonstrates a minimal, reusable pattern you can adapt to other providers.

Follow the instructions in the companion `README_CHAT_INTEGRATION.md` to install dependencies and set your API key.

## Prerequisites
- Set the environment variable `OPENAI_API_KEY` with your API key.
- Optionally set `OPENAI_CHAT_MODEL` to select a model (defaults to `gpt-4o-mini`).
- Install requirements (`pip install -r requirements.txt`).

In [ ]:
import os
import openai
from IPython.display import display
import ipywidgets as widgets

# Configure API key and model via environment variables
openai.api_key = os.getenv('OPENAI_API_KEY')
MODEL = os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini')

history = [{'role': 'system', 'content': 'You are a helpful assistant integrated with Jupyter.'}]

def chat_call(messages, model=MODEL):
    if openai.api_key is None:
        raise RuntimeError('OPENAI_API_KEY not set in environment')
    resp = openai.ChatCompletion.create(model=model, messages=messages)
    return resp['choices'][0]['message']['content']

# Quick test helper (uncomment to run a test call)
# print(chat_call(history, MODEL))

In [ ]:
# Interactive widget UI
input_box = widgets.Textarea(placeholder='Type a message...', layout=widgets.Layout(width='100%', height='120px'))
send_btn = widgets.Button(description='Send')
output = widgets.Output()
save_btn = widgets.Button(description='Save conversation')

def on_send(b):
    user_text = input_box.value.strip()
    if not user_text:
        return
    history.append({'role': 'user', 'content': user_text})
    output.clear_output()
    with output:
        print('Assistant is thinking...')
    try:
        resp = chat_call(history)
    except Exception as e:
        output.clear_output()
        with output:
            print('Error:', e)
        return
    history.append({'role': 'assistant', 'content': resp})
    output.clear_output()
    with output:
        print(resp)

send_btn.on_click(on_send)

def on_save(b):
    fname = 'jupyter_chat_history.txt'
    with open(fname, 'w', encoding='utf-8') as f:
        for m in history:
            f.write(f"{m['role'].upper()}: {m['content']}\n\n")
    with output:
        print('Saved to', fname)

save_btn.on_click(on_save)

display(widgets.VBox([input_box, widgets.HBox([send_btn, save_btn]), output]))

**Notes**:
- You can change the target model by setting `OPENAI_CHAT_MODEL` in the environment.
- The widget will append the conversation to `history`. Use the Save button to persist to a text file.